# FuseMap Agent — Conversational AI for the Mouse Brain Atlas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wanglab-broad/FuseMap/blob/main/docs/notebooks/agent_colab.ipynb)

This notebook runs the **FuseMap Agent**, a multi-agent conversational AI for exploring the 3D mouse brain atlas (molCCF) and analyzing spatial transcriptomics data — entirely from your browser, with **no GPU and no local installation**.

The supervisor agent (the same wiring as the Streamlit app, `app.py`) delegates to three expert agents:

- **ResearchAgent** — literature search on diseases and conditions (via Tavily web search)
- **AtlasAgent** — queries the 3D mouse brain atlas: brain regions, cell types, gene expression, and matching 2D tissue sections
- **FuseMapAgent** — runs FuseMap analysis workflows on your own spatial transcriptomics data

**What you need**

- An **OpenAI API key** (required — powers all agents)
- A **Tavily API key** (optional; free at https://www.tavily.com/ — enables the literature ResearchAgent, which is skipped if no key is provided)


## 1. Install

Clone the repository and install dependencies. The pinned `requirements.txt` is tried first; if it cannot be resolved on the current Colab image, a best-effort minimal dependency set is installed instead.


In [ ]:
!git clone -q https://github.com/wanglab-broad/FuseMap.git
%cd FuseMap
!pip install -q -r requirements.txt 2>/dev/null || pip install -q langchain==0.3.25 langchain-community langchain-openai langchain-anthropic langgraph tavily-python duckduckgo-search easydict scanpy plotly dgl streamlit audio-recorder-streamlit


## 2. Download the atlas data

The atlas tools load reference data from `agent_setup/atlas_data/` at import time, so the data must be in place **before** building the agent. **`ad_cell.h5ad` is required** (the single-cell-resolution atlas backing all brain region / cell type / section queries), alongside `ad_gene.h5ad` and the lookup CSVs that ship with the repository. See the "Atlas molCCF data" entry in `agent_setup/README.md` for the download link.


In [ ]:
# TODO(maintainer): fill in the Google Drive folder ID for agent atlas data
# !gdown --folder <ATLAS_DATA_DRIVE_ID> -O agent_setup/atlas_data
# (See also the "Atlas molCCF data" Google Drive link in agent_setup/README.md.)

import os

if os.path.exists("agent_setup/atlas_data/ad_cell.h5ad"):
    print("Atlas data found: agent_setup/atlas_data/ad_cell.h5ad")
else:
    print("Atlas data NOT found. ad_cell.h5ad is required by the atlas tools —")
    print("uncomment and fill in the gdown command above, then re-run this cell.")


## 3. API keys

Keys are read with `getpass`, so they are never shown or stored in the notebook. Press Enter to skip the optional Tavily key.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key (required): ")

if not os.environ.get("TAVILY_API_KEY"):
    _tavily = getpass("Tavily API key (optional, press Enter to skip): ").strip()
    if _tavily:
        os.environ["TAVILY_API_KEY"] = _tavily
    else:
        print("No Tavily key — the literature ResearchAgent will be disabled.")


## 4. Build the supervisor agent

This mirrors the Streamlit app's wiring in `app.py`: `create_llm` + the three agent tools + `supervisor_agent_prompt` + `ConversationBufferMemory`, assembled with LangChain's `initialize_agent` (OpenAI-functions agent). Two differences from the app:

- a plain in-memory chat history is used instead of `StreamlitChatMessageHistory`;
- the ResearchAgent is only added when a Tavily key is present.

The agent tool modules reference Streamlit for progress bars; outside `streamlit run` those calls are harmless no-ops (their warnings are silenced below).


In [ ]:
import os
import datetime
import logging

# The agent tool modules use Streamlit progress bars; outside `streamlit run`
# these are harmless no-ops. Silence the "missing ScriptRunContext" warnings.
logging.getLogger("streamlit").setLevel(logging.ERROR)
logging.getLogger(
    "streamlit.runtime.scriptrunner_utils.script_run_ctx"
).setLevel(logging.ERROR)

# Output folders used by the agent tools (mirrors app.py)
save_path = "./output/"
os.makedirs(f"{save_path}/data", exist_ok=True)
os.makedirs(f"{save_path}/figures", exist_ok=True)
os.makedirs(f"{save_path}/fusemap", exist_ok=True)

from langchain.agents import AgentType, initialize_agent
from langchain.memory import ConversationBufferMemory
from langchain.prompts import MessagesPlaceholder
from langchain.schema.messages import SystemMessage

from agent_setup.config import create_llm
from agent_setup.prompt import supervisor_agent_prompt
from agent_setup.agents.atlas_agent import atlas_agent_tool
from agent_setup.agents.fusemap_agent import fusemap_agent_tool

# LLM — same default model choice as app.py
llm = create_llm(model_choice="gpt-4o", api_key=os.environ["OPENAI_API_KEY"])

# Conversation memory (plain in-memory history instead of StreamlitChatMessageHistory)
memory = ConversationBufferMemory(
    return_messages=True, memory_key="memory", output_key="output"
)

# System message: supervisor prompt + current time (as in app.py)
current_time = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9)))
current_time_str = current_time.strftime("%Y-%m-%d %H:%M:%S %Z%z")

content = f"""{supervisor_agent_prompt}
Current Time: {current_time_str}.
"""

agent_kwargs = {
    "extra_prompt_messages": [MessagesPlaceholder(variable_name="memory")],
    "system_message": SystemMessage(content=content),
}

# Tools — the ResearchAgent requires a Tavily key and is skipped when absent
tools = []
if os.environ.get("TAVILY_API_KEY"):
    from agent_setup.agents.research_agent import research_agent_tool
    tools.append(research_agent_tool(llm))
else:
    print("TAVILY_API_KEY not set — building the agent without the ResearchAgent.")
tools.append(atlas_agent_tool(llm))
tools.append(fusemap_agent_tool(llm))

supervisor_agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    agent_kwargs=agent_kwargs,
    memory=memory,
    verbose=True,
)

print("Supervisor agent ready with tools:", [t.name for t in tools])


## 5. Chat with the agent

Run the next cell and type your question at the prompt (type `exit` or `quit` to stop). Example questions:

- *"Which brain regions express Slc17a7?"*
- *"What cell types are in the hippocampus CA1 region?"*
- *"Search literature about how cell states change in mouse hippocampus with Alzheimer's disease."* (requires the Tavily key)

Generated figures (3D region/gene plots, 2D section plots) are saved under `output/figures/` — browse and download them from the Colab file panel on the left.


In [ ]:
# Chat with the agent. Type "exit" or "quit" to stop.
while True:
    q = input("You: ")
    if q.strip().lower() in ("exit", "quit"):
        break
    print("Agent:", supervisor_agent.run(input=q))


---

## Learn more

- FuseMap documentation and tutorials: https://fusemap.readthedocs.io
- Hosted Brain Spatial Atlas: https://www.spatial-atlas.net/FuseMap/
- Source code: https://github.com/wanglab-broad/FuseMap
